# Control Feature Visualization
Distributions of lexical, syntactic, acoustic, and surprisal features across all patients.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

FEATURES_DIR  = '/scratch/aniluchavez/ConvoDATAS/ControlFeatures'
SURPRISAL_DIR = '/scratch/aniluchavez/ConvoDATAS/Surprisal'

# load all patients
dfs = []
for fpath in sorted(glob.glob(os.path.join(FEATURES_DIR, '*_control_features.csv'))):
    pid = os.path.basename(fpath).replace('_control_features.csv', '')
    df  = pd.read_csv(fpath)
    df['patient'] = pid

    # merge surprisal
    surp_path = os.path.join(SURPRISAL_DIR, f'{pid}_surprisal.csv')
    if os.path.exists(surp_path):
        surp = pd.read_csv(surp_path)[['surprisal']]
        if len(surp) == len(df):
            df['surprisal'] = surp['surprisal'].values
        else:
            df['surprisal'] = np.nan
    else:
        df['surprisal'] = np.nan

    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(dfs)} patients, {len(data)} total words')
print('Columns:', data.columns.tolist())

## Lexical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Lexical Features', fontsize=14, fontweight='bold')

# global word frequency
ax = axes[0, 0]
for pid, grp in data.groupby('patient'):
    grp['log_word_freq'].dropna().plot.kde(ax=ax, alpha=0.4, linewidth=1.2)
data['log_word_freq'].dropna().plot.kde(ax=ax, color='black', linewidth=2.5, label='all')
ax.set_xlabel('Log Word Frequency (Zipf)')
ax.set_title('Global Word Frequency\n(English corpus)')
ax.set_xlim(0, 8)

# local repetition rate
ax = axes[0, 1]
for pid, grp in data.groupby('patient'):
    grp['local_rate'].dropna().plot.kde(ax=ax, alpha=0.4, linewidth=1.2)
data['local_rate'].dropna().plot.kde(ax=ax, color='black', linewidth=2.5, label='all')
ax.set_xlabel('Local Repetition Rate (prior occ. / position)')
ax.set_title('Local Word Frequency\n(within conversation)')
ax.set_xlim(0, 0.5)

# global vs local scatter
ax = axes[0, 2]
sample = data.dropna(subset=['log_word_freq', 'local_rate']).sample(min(6000, len(data)), random_state=42)
ax.scatter(sample['log_word_freq'], sample['local_rate'],
           alpha=0.12, s=6, color='mediumpurple')
r = sample[['log_word_freq', 'local_rate']].corr().iloc[0, 1]
ax.set_xlabel('Global Freq (Zipf)')
ax.set_ylabel('Local Rate')
ax.set_title(f'Global vs Local Frequency\n(r = {r:.2f})')

# word length
ax = axes[1, 0]
data['word_length'].dropna().value_counts().sort_index().plot.bar(ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Word Length (chars)')
ax.set_title('Word Length')
ax.set_xlim(-1, 20)

# word duration
ax = axes[1, 1]
for pid, grp in data.groupby('patient'):
    grp['Duration'].dropna().clip(upper=1000).plot.kde(ax=ax, alpha=0.4, linewidth=1.2)
data['Duration'].dropna().clip(upper=1000).plot.kde(ax=ax, color='black', linewidth=2.5, label='all')
ax.set_xlabel('Duration (ms)')
ax.set_title('Word Duration')

# local count histogram
ax = axes[1, 2]
data['local_count'].dropna().clip(upper=30).plot.hist(ax=ax, bins=31, color='coral', alpha=0.8, edgecolor='white')
ax.set_xlabel('Prior occurrences in conversation')
ax.set_title('Local Repetition Count')

plt.tight_layout()
plt.show()

## Part of Speech Distribution

In [ ]:
pos_data = data.dropna(subset=['POS'])
if pos_data.empty:
    print('No POS data (only available for the 10 patients with withPOS files)')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Part of Speech', fontsize=14, fontweight='bold')

    # overall POS distribution
    counts = pos_data['POS'].value_counts().head(15)
    counts.plot.bar(ax=axes[0], color='mediumseagreen', alpha=0.85, edgecolor='white')
    axes[0].set_xlabel('POS tag')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Top 15 POS Tags (all patients)')
    axes[0].tick_params(axis='x', rotation=45)

    # dep depth by POS
    top_pos = counts.index[:8].tolist()
    sub = pos_data[pos_data['POS'].isin(top_pos)]
    sns.boxplot(data=sub, x='POS', y='dep_depth', order=top_pos,
                palette='Set2', ax=axes[1], fliersize=2)
    axes[1].set_title('Dependency Depth by POS Tag')
    axes[1].set_xlabel('POS tag')
    axes[1].set_ylabel('dep_depth')

    plt.tight_layout()
    plt.show()

## Dependency Labels

In [ ]:
dep_data = data[data['dep_label'].notna() & (data['dep_label'] != '')]

top_n = 20
label_counts = dep_data['dep_label'].value_counts().head(top_n)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Dependency Labels', fontsize=14, fontweight='bold')

# --- label frequency bar chart ---
ax = axes[0]
label_counts.plot.bar(ax=ax, color='steelblue', alpha=0.85, edgecolor='white')
ax.set_xlabel('Dependency Label')
ax.set_ylabel('Count')
ax.set_title(f'Top {top_n} Dep Labels (all patients)')
ax.tick_params(axis='x', rotation=45)

# --- dep depth by label (top 12) ---
ax = axes[1]
top12 = label_counts.index[:12].tolist()
sns.boxplot(data=dep_data[dep_data['dep_label'].isin(top12)],
            x='dep_label', y='dep_depth', order=top12,
            palette='Set3', ax=ax, fliersize=1.5)
ax.set_xlabel('Dependency Label')
ax.set_ylabel('Depth in Tree')
ax.set_title('Tree Depth by Dep Label')
ax.tick_params(axis='x', rotation=45)

# --- POS x dep_label heatmap (top labels & POS tags) ---
ax = axes[2]
pos_dep = dep_data.dropna(subset=['POS'])
if not pos_dep.empty:
    top_pos    = pos_dep['POS'].value_counts().head(10).index.tolist()
    top_labels = pos_dep['dep_label'].value_counts().head(12).index.tolist()
    sub = pos_dep[pos_dep['POS'].isin(top_pos) & pos_dep['dep_label'].isin(top_labels)]
    ct = pd.crosstab(sub['POS'], sub['dep_label'])
    ct_norm = ct.div(ct.sum(axis=1), axis=0)  # row-normalize
    sns.heatmap(ct_norm, cmap='YlOrRd', ax=ax, annot=False,
                linewidths=0.3, cbar_kws={'label': 'proportion'})
    ax.set_title('POS × Dep Label\n(row-normalized)')
    ax.set_xlabel('Dependency Label')
    ax.set_ylabel('POS Tag')
    ax.tick_params(axis='x', rotation=45)
else:
    ax.text(0.5, 0.5, 'No POS data available', ha='center', va='center')
    ax.set_title('POS × Dep Label')

plt.tight_layout()
plt.show()

## Surprisal

In [ ]:
surp_data = data.dropna(subset=['surprisal'])
if surp_data.empty:
    print('No surprisal data found.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Surprisal (GPT-2)', fontsize=14, fontweight='bold')

    # distribution per patient
    for pid, grp in surp_data.groupby('patient'):
        grp['surprisal'].clip(upper=30).plot.kde(ax=axes[0], alpha=0.4, linewidth=1.2)
    surp_data['surprisal'].clip(upper=30).plot.kde(ax=axes[0], color='black', linewidth=2.5)
    axes[0].set_xlabel('Surprisal (bits)')
    axes[0].set_title('Surprisal Distribution')

    # surprisal vs word frequency
    sample = surp_data.dropna(subset=['log_word_freq']).sample(min(5000, len(surp_data)), random_state=42)
    axes[1].scatter(sample['log_word_freq'], sample['surprisal'].clip(upper=30),
                    alpha=0.15, s=8, color='steelblue')
    axes[1].set_xlabel('Log Word Frequency (Zipf)')
    axes[1].set_ylabel('Surprisal (bits)')
    axes[1].set_title('Surprisal vs Word Frequency')

    plt.tight_layout()
    plt.show()

## Acoustic Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Acoustic Features', fontsize=14, fontweight='bold')

# pitch (f0)
ax = axes[0]
for pid, grp in data.groupby('patient'):
    grp['f0_mean'].dropna().clip(50, 500).plot.kde(ax=ax, alpha=0.4, linewidth=1.2)
data['f0_mean'].dropna().clip(50, 500).plot.kde(ax=ax, color='black', linewidth=2.5)
ax.set_xlabel('Mean F0 (Hz)')
ax.set_title('Pitch (voiced words only)')

# envelope (RMS)
ax = axes[1]
for pid, grp in data.groupby('patient'):
    grp['rms_mean'].dropna().clip(upper=grp['rms_mean'].quantile(0.99)).plot.kde(
        ax=ax, alpha=0.4, linewidth=1.2)
ax.set_xlabel('RMS Amplitude')
ax.set_title('Envelope (RMS)')

# spectral flux
ax = axes[2]
for pid, grp in data.groupby('patient'):
    vals = grp['spectral_flux'].dropna()
    if len(vals) > 10:
        vals.clip(upper=vals.quantile(0.99)).plot.kde(ax=ax, alpha=0.4, linewidth=1.2)
data['spectral_flux'].dropna().clip(
    upper=data['spectral_flux'].quantile(0.99)).plot.kde(
    ax=ax, color='black', linewidth=2.5)
ax.set_xlabel('Spectral Flux')
ax.set_title('Spectral Flux')

plt.tight_layout()
plt.show()

## Acoustic Features by Patient (violin)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 13))
fig.suptitle('Acoustic Features by Patient', fontsize=14, fontweight='bold')

short_pid = {p: p[:5] for p in data['patient'].unique()}
plot_data = data.copy()
plot_data['pid_short'] = plot_data['patient'].map(short_pid)

order = sorted(plot_data['pid_short'].unique())

sns.violinplot(data=plot_data.dropna(subset=['f0_mean']),
               x='pid_short', y='f0_mean', order=order,
               palette='husl', ax=axes[0], cut=0, inner='box')
axes[0].set_ylabel('Mean F0 (Hz)')
axes[0].set_title('Pitch per Patient')
axes[0].set_xlabel('')

sns.violinplot(data=plot_data.dropna(subset=['rms_mean']),
               x='pid_short', y='rms_mean', order=order,
               palette='husl', ax=axes[1], cut=0, inner='box')
axes[1].set_ylabel('RMS Amplitude')
axes[1].set_title('Envelope per Patient')
axes[1].set_xlabel('')

flux_clip = plot_data['spectral_flux'].quantile(0.99)
sns.violinplot(data=plot_data.dropna(subset=['spectral_flux']).assign(
                   spectral_flux=lambda d: d['spectral_flux'].clip(upper=flux_clip)),
               x='pid_short', y='spectral_flux', order=order,
               palette='husl', ax=axes[2], cut=0, inner='box')
axes[2].set_ylabel('Spectral Flux')
axes[2].set_title('Spectral Flux per Patient')
axes[2].set_xlabel('Patient')

plt.tight_layout()
plt.show()

## Feature Correlation Matrix

In [ ]:
feat_cols = ['log_word_freq', 'word_length', 'Duration', 'local_count',
             'dep_depth', 'speaking_rate',
             'f0_mean', 'rms_mean', 'spectral_flux', 'surprisal']
feat_cols = [c for c in feat_cols if c in data.columns]

corr = data[feat_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()